<a href="https://colab.research.google.com/github/boss-defender/Smart-Fine-Tune/blob/main/SmartFineTuner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#🦾 FineTune
**🤝 It will free you from hassle while fine tuning ai model.**

**🎯 Check Github Repo here:**

**📍 https://github.com/boss-defender/Smart-Fine-Tune**

**⚡ Powered by [unslothai/unsloth](https://github.com/unslothai/unsloth).**

#🤔 Beginner ? Don't worry .

**🌐 Just give Model path and Dataset path from hugging face or Google Drive.**

**⚙️ Set max sample minimum 100. But no input in max sample will lead this code to use whole dataset.**

**💡 Keep other box empty as it is.**

**🛠️ If you have technical knowledge , fill up those box carefully**

**✨ Click run & see the magic.**

#⚠️ Caution:

**💰 With Colab free tier , you can only finetune smaller Model with smaller Datasets or smaller Max_Samples.**

**✍️ For a complex dataset structure, you may need to make minor manual edits to the code.**



# 💡 Want to train on your own dataset?

👉 First, share this Colab notebook by selecting **`Share → General access → Anyone with the link`**. Then upload **your dataset and this Colab notebook** to an AI assistant such as **ChatGPT, Gemini, or Claude**.

Tell the AI:

> **“Please convert and format my custom dataset so that it is fully compatible with the dataset format expected by Cell 1 of this Colab notebook and ready for training.”**

✅ The AI can then restructure your dataset into the correct format for this fine-tuning system.


In [ ]:
# @title ⬇🚀 1. Universal Smart Fine Tuning (One-Click Auto Fine-Tuner)
# ==============================================================================
# ⚡ UNIVERSAL PRODUCTION MODEL & DATASET CAPABILITY ROUTER
# Robust Pre-Load Inspection | Math Step Calculation | Fault-Tolerant Export
# ==============================================================================

# --- 🎛️ CONFIGURATION & FORM INPUTS ---
MODEL_NAME = "Qwen/Qwen3-1.7B"           #@param {type:"string"}
DATASET_NAME = "bespokelabs/Bespoke-Stratos-17k"                  #@param {type:"string"}
DATASET_CONFIG = ""                                #@param {type:"string"}
NUM_EPOCHS = 1.0                                   #@param {type:"number"}
MAX_SAMPLES = "100"                                  #@param {type:"string"}
MAX_SEQ_LENGTH = ""                               #@param {type:"string"}
LOAD_IN_4BIT = True                                #@param {type:"boolean"}
HF_TOKEN = ""                                      #@param {type:"string"}

# 📊 W&B — disabled by default
ENABLE_WANDB = False                            #@param {type:"boolean"}

# Advanced Overrides
LEARNING_RATE = ""                              #@param {type:"string"}
MAX_STEPS = ""                                  #param {type:"string"}
SAVE_STEPS = "5"                                #@param {type:"string"}


# ==============================================================================
# 📊 W&B CONTROL
# ==============================================================================

import os

if ENABLE_WANDB:
    os.environ.pop("WANDB_DISABLED", None)
    os.environ["WANDB_MODE"] = "online"
else:
    os.environ["WANDB_DISABLED"] = "true"
    os.environ["WANDB_MODE"] = "disabled"

# --- STEP 1: MOUNT GOOGLE DRIVE & PERSISTENT STORAGE HIERARCHY ---
print("📂 [1/10] Connecting Google Drive for Checkpoints & Export...")
DRIVE_BASE_DIR = "/content/drive/MyDrive/unsloth_checkpoints"
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print("✅ Google Drive connected successfully!")
except Exception as e:
    print(f"⚠️ Google Drive not mounted ({e}). Using persistent local storage /content/checkpoints")
    DRIVE_BASE_DIR = "/content/checkpoints"

# --- STEP 2: STABLE DEPENDENCY STACK & VERSION VERIFICATION ---
# Note: Executed BEFORE importing C-extension modules (torch, unsloth, etc.) to prevent C-API memory flags corruption.
print("🔄 [2/10] Verifying Dependency Stack & Core Versions...")
!pip install --quiet --upgrade pip wheel
!pip install --quiet "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" unsloth_zoo
!pip install --quiet "pyarrow>=15.0.0" "transformers>=4.51.3,<5.0.0" "datasets>=3.4.1,<4.0.0" "peft>=0.18.0,<1.0.0" "trl>=0.18.2,<=0.24.0" "accelerate>=1.0.0,<2.0.0" "bitsandbytes>=0.43.1" huggingface_hub hf_transfer "setuptools<82"

import sys

# Guard: Catch PyArrow binary ABI mismatch on Colab live update & trigger automatic runtime restart
try:
    import pyarrow.dataset
except ValueError as e_pa:
    if "size changed" in str(e_pa) or "binary incompatibility" in str(e_pa):
        print("⚠️ PyArrow C-extension binary update detected in running kernel. Auto-restarting Colab runtime...")
        import os
        os._exit(0)

import math
import hashlib
import json
import shutil

# Import Unsloth FIRST before torch/transformers/datasets/trl as required by Unsloth optimization engine
try:
    import unsloth
except Exception as e_unsloth:
    print(f"⚠️ Unsloth import notice: {e_unsloth}")

import torch
import transformers
import datasets
import peft
import trl
import accelerate

print(f"✅ Verified Versions: Transformers={transformers.__version__}, Datasets={datasets.__version__}, PEFT={peft.__version__}, TRL={trl.__version__}, Accelerate={accelerate.__version__}\n")

# --- STEP 2: HARDWARE DETECTION & AUTOMATIC TUNING ---
print("🔍 [3/10] Verifying Hardware & Auto-Tuning Parameters...")
if not torch.cuda.is_available():
    raise SystemError("❌ No GPU found! Please change Colab Runtime to T4 GPU or higher (Runtime > Change runtime type > GPU).")

gpu_name = torch.cuda.get_device_name(0)
total_vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
has_bf16 = torch.cuda.is_bf16_supported()
print(f"✅ GPU Detected: {gpu_name} ({total_vram:.2f} GB VRAM, BF16 Supported: {has_bf16})")

# Auto-calculate micro-batch size and gradient accumulation based on VRAM
if total_vram < 12:
    BATCH_SIZE = 1
    GRAD_ACCUM = 8
elif total_vram < 20:
    BATCH_SIZE = 2
    GRAD_ACCUM = 4
elif total_vram < 40:
    BATCH_SIZE = 4
    GRAD_ACCUM = 2
else:
    BATCH_SIZE = 8
    GRAD_ACCUM = 1

print(f"⚡ Hardware Auto-Tuned: Micro-Batch Size = {BATCH_SIZE}, Grad Accumulation = {GRAD_ACCUM} (Effective Batch = {BATCH_SIZE * GRAD_ACCUM})\n")



# --- STEP 4: HARDENED PRE-LOAD MODEL CAPABILITY CLASSIFIER ---
print("👁️ [4/10] Inspecting Model Capability Route...")

def inspect_model_capabilities_hardened(model_name, token=None):
    m_lower = model_name.lower()

    # 1. Immediate keyword checks for excluded modalities
    diffusion_keywords = ["stable-diffusion", "diffusers", "sdxl", "flux", "midjourney", "dall-e", "imagen", "latent-diffusion"]
    if any(k in m_lower for k in diffusion_keywords):
        return "UNSUPPORTED_DIFFUSION", "Model ID matches image generation / diffusion framework"

    video_keywords = ["open-sora", "cogvideo", "svd", "stable-video", "animatediff", "videogen"]
    if any(k in m_lower for k in video_keywords):
        return "UNSUPPORTED_VIDEO", "Model ID matches video generation framework"

    classification_keywords = ["finetuned-sst", "sequence-classification", "for-sequence-classification", "sentence-transformers"]
    if any(k in m_lower for k in classification_keywords):
        return "UNSUPPORTED_CLASSIFICATION", "Model ID string matches sequence classification / embedding model"

    # 2. Config Metadata & Architecture Inspection via AutoConfig
    try:
        from transformers import AutoConfig
        cfg = AutoConfig.from_pretrained(model_name, token=token if token else None, trust_remote_code=True)

        model_type = str(getattr(cfg, "model_type", "")).lower()
        archs = [str(a).lower() for a in getattr(cfg, "architectures", [])]
        arch_str = " ".join(archs)

        if any(k in model_type or k in arch_str for k in ["diffusion", "unet", "sdxl", "flux"]):
            return "UNSUPPORTED_DIFFUSION", f"Config indicates diffusion model (type='{model_type}', arch='{arch_str}')"

        if any(k in model_type or k in arch_str for k in ["sora", "cogvideo", "videodiffusion"]):
            return "UNSUPPORTED_VIDEO", f"Config indicates video generation model (type='{model_type}', arch='{arch_str}')"

        if any("sequenceclassification" in a or "fortokenclassification" in a or "embedding" in a for a in archs):
            return "UNSUPPORTED_CLASSIFICATION", f"Config indicates classification/embedding architecture ({archs})"

        has_vision_attr = any(hasattr(cfg, attr) for attr in ["vision_config", "is_visual", "image_token_id", "visual", "vision_feature_layer"])
        vlm_patterns = ["vision", "vl", "llava", "paligemma", "blip", "idefics", "qwen2_vl", "qwen3_vl", "florence", "vision2seq", "conditionalgeneration", "phi3v"]

        is_vlm_type = any(p in model_type for p in vlm_patterns)
        is_vlm_arch = any(p in a for a in archs for p in vlm_patterns)

        if has_vision_attr or is_vlm_type or is_vlm_arch:
            return "VISION_LANGUAGE", f"Model Config (model_type='{model_type}', arch={archs})"

        is_enc_dec = getattr(cfg, "is_encoder_decoder", False)
        if is_enc_dec or any(p in model_type or p in arch_str for p in ["t5", "bart", "pegasus", "marian", "encoderdecoder"]):
            return "SEQ2SEQ_LM", f"Encoder-Decoder / Seq2Seq Model Config (model_type='{model_type}')"

        if any("causallm" in a or "lmheadmodel" in a for a in archs) or model_type in ["llama", "qwen2", "qwen2_5", "qwen3", "qwen3_moe", "gemma", "gemma2", "mistral", "mixtral", "phi", "phi3", "phi4", "starcoder2", "deepseek_v2", "deepseek_v3"]:
            return "CAUSAL_LM", f"Causal LM Model Config (model_type='{model_type}')"

        return "CAUSAL_LM", f"Standard Generative LM Config (model_type='{model_type}')"

    except Exception as e_cfg:
        err_str = str(e_cfg).lower()
        if any(k in err_str for k in ["qwen3", "qwen", "llama", "gemma", "mistral", "phi", "deepseek"]):
            return "CAUSAL_LM", f"Detected recognized LLM architecture in error message ({e_cfg})"

        vlm_keywords = ["vision", "vl", "llava", "paligemma", "blip", "idefics", "qwen2_vl", "florence", "vision2seq"]
        if any(k in m_lower for k in vlm_keywords):
            return "VISION_LANGUAGE", "Fallback String Match: VLM keywords in model ID"

        seq2seq_keywords = ["t5", "flan-t5", "bart", "pegasus"]
        if any(k in m_lower for k in seq2seq_keywords):
            return "SEQ2SEQ_LM", "Fallback String Match: Seq2Seq keywords in model ID"

        return "CAUSAL_LM", f"Fallback Default: Causal LM string assumption ({e_cfg})"

model_route, route_reason = inspect_model_capabilities_hardened(MODEL_NAME, HF_TOKEN)
print(f"🎯 Model Capability Route: {model_route} [{route_reason}]")

if model_route.startswith("UNSUPPORTED"):
    raise SystemExit(f"🛑 UNSUPPORTED MODEL ARCHITECTURE ({model_route}): {route_reason}. Fine-tuner safely stopped to prevent invalid training.")

is_vision_model = (model_route == "VISION_LANGUAGE")

# Auto Learning Rate
try:
    lr_val = float(LEARNING_RATE) if LEARNING_RATE is not None and str(LEARNING_RATE).strip() != "" else 2e-4
    if lr_val <= 0: lr_val = 2e-4
except Exception:
    lr_val = 2e-4
LEARNING_RATE_RESOLVED = lr_val

# Auto Max Seq Length
if is_vision_model:
    MAX_SEQ_LENGTH_RESOLVED = None
    print("📏 Max Sequence Length: Set to None (TRL VLM Recommendation - Preserves visual patch tokens)")
else:
    try:
        seq_len_val = int(MAX_SEQ_LENGTH) if MAX_SEQ_LENGTH is not None and str(MAX_SEQ_LENGTH).strip() != "" else 2048
        if seq_len_val <= 0: seq_len_val = 2048
    except Exception:
        seq_len_val = 2048
    MAX_SEQ_LENGTH_RESOLVED = seq_len_val
    print(f"📏 Max Sequence Length: Set to {MAX_SEQ_LENGTH_RESOLVED}")

# --- STEP 5: BULLETPROOF DATASET LOADING & REPOSITORY FALLBACK ---
print(f"\n📊 [5/10] Loading Dataset: '{DATASET_NAME}'...")
from datasets import load_dataset, get_dataset_config_names
from huggingface_hub import list_repo_files

def bulletproof_load_dataset(dataset_name, dataset_config=None, token=None):
    """
    Universal dataset loader.

    Supports:
      1. Hugging Face dataset IDs
         Example:
             bespokelabs/Bespoke-Stratos-17k

      2. Local Google Drive / Colab directories
         Example:
             /content/drive/MyDrive/Datasets/MyDataset

      3. Direct local dataset files
         Example:
             /content/drive/MyDrive/Datasets/train.jsonl
             /content/drive/MyDrive/Datasets/train.parquet
             /content/drive/MyDrive/Datasets/train.csv
             /content/drive/MyDrive/Datasets/train.txt
    """

    dataset_name = str(dataset_name).strip()

    if not dataset_name:
        raise ValueError("❌ DATASET_NAME cannot be empty.")

    # ==========================================================================
    # 🟢 MODE 1 — LOCAL DATASET
    # ==========================================================================

    if os.path.exists(dataset_name):

        print("📁 Local dataset path detected.")
        print(f"📂 Dataset source: {dataset_name}")

        # ----------------------------------------------------------------------
        # Direct file
        # ----------------------------------------------------------------------

        if os.path.isfile(dataset_name):

            ext = dataset_name.lower()

            print(f"📄 Local dataset file: {dataset_name}")

            if ext.endswith((".json", ".jsonl", ".jsonl.zst")):

                print("🔄 Loading local JSON / JSONL dataset...")

                return load_dataset(
                    "json",
                    data_files={"train": dataset_name}
                )

            elif ext.endswith(".parquet"):

                print("🔄 Loading local Parquet dataset...")

                return load_dataset(
                    "parquet",
                    data_files={"train": dataset_name}
                )

            elif ext.endswith(".csv"):

                print("🔄 Loading local CSV dataset...")

                return load_dataset(
                    "csv",
                    data_files={"train": dataset_name}
                )

            elif ext.endswith((".txt", ".text")):

                print("🔄 Loading local text dataset...")

                return load_dataset(
                    "text",
                    data_files={"train": dataset_name}
                )

            else:

                raise ValueError(
                    "❌ Unsupported local dataset file type.\n\n"
                    f"File: {dataset_name}\n\n"
                    "Supported types:\n"
                    "  • .json\n"
                    "  • .jsonl\n"
                    "  • .jsonl.zst\n"
                    "  • .parquet\n"
                    "  • .csv\n"
                    "  • .txt"
                )

        # ----------------------------------------------------------------------
        # Local directory
        # ----------------------------------------------------------------------

        if os.path.isdir(dataset_name):

            print("📂 Local dataset directory detected.")
            print("🔎 Searching for supported dataset files...")

            supported_files = []

            for root, dirs, files in os.walk(dataset_name):

                for filename in files:

                    lower_name = filename.lower()

                    if lower_name.endswith(
                        (
                            ".json",
                            ".jsonl",
                            ".jsonl.zst",
                            ".parquet",
                            ".csv",
                            ".txt",
                            ".text",
                        )
                    ):

                        supported_files.append(
                            os.path.join(root, filename)
                        )

            if not supported_files:

                raise RuntimeError(
                    "❌ No supported dataset files were found inside:\n"
                    f"{dataset_name}\n\n"
                    "Supported:\n"
                    "  • JSON / JSONL\n"
                    "  • Parquet\n"
                    "  • CSV\n"
                    "  • TXT"
                )

            supported_files.sort()

            print(
                f"✅ Found {len(supported_files)} supported dataset file(s)."
            )

            for file_path in supported_files:
                print(f"   📄 {file_path}")

            # ------------------------------------------------------------------
            # Detect file family
            # ------------------------------------------------------------------

            extensions = set()

            for file_path in supported_files:

                lower_name = file_path.lower()

                if lower_name.endswith(
                    (".json", ".jsonl", ".jsonl.zst")
                ):
                    extensions.add("json")

                elif lower_name.endswith(".parquet"):
                    extensions.add("parquet")

                elif lower_name.endswith(".csv"):
                    extensions.add("csv")

                elif lower_name.endswith(
                    (".txt", ".text")
                ):
                    extensions.add("text")

            # ------------------------------------------------------------------
            # Prevent accidentally mixing incompatible file formats
            # ------------------------------------------------------------------

            if len(extensions) > 1:

                raise ValueError(
                    "❌ Multiple incompatible dataset formats detected "
                    "inside the local dataset directory.\n\n"
                    f"Detected formats: {sorted(extensions)}\n\n"
                    "Please keep one dataset format per directory."
                )

            dataset_type = next(iter(extensions))

            # ------------------------------------------------------------------
            # Load local JSON / JSONL
            # ------------------------------------------------------------------

            if dataset_type == "json":

                print("🔄 Loading local JSON / JSONL dataset...")

                return load_dataset(
                    "json",
                    data_files={
                        "train": supported_files
                    }
                )

            # ------------------------------------------------------------------
            # Load local Parquet
            # ------------------------------------------------------------------

            if dataset_type == "parquet":

                print("🔄 Loading local Parquet dataset...")

                return load_dataset(
                    "parquet",
                    data_files={
                        "train": supported_files
                    }
                )

            # ------------------------------------------------------------------
            # Load local CSV
            # ------------------------------------------------------------------

            if dataset_type == "csv":

                print("🔄 Loading local CSV dataset...")

                return load_dataset(
                    "csv",
                    data_files={
                        "train": supported_files
                    }
                )

            # ------------------------------------------------------------------
            # Load local TXT
            # ------------------------------------------------------------------

            if dataset_type == "text":

                print("🔄 Loading local text dataset...")

                return load_dataset(
                    "text",
                    data_files={
                        "train": supported_files
                    }
                )

        raise RuntimeError(
            f"❌ Local dataset path could not be loaded:\n{dataset_name}"
        )

    # ==========================================================================
    # ☁️ MODE 2 — HUGGING FACE DATASET
    # ==========================================================================

    print("☁️ Hugging Face dataset ID detected.")
    print(f"📊 Dataset source: {dataset_name}")

    cfg = (
        str(dataset_config).strip()
        if dataset_config is not None
        else ""
    )

    if cfg.lower() in [
        "",
        "auto",
        "none",
        "default",
        "null"
    ]:
        cfg = None

    # --------------------------------------------------------------------------
    # Direct config load
    # --------------------------------------------------------------------------

    if cfg:

        try:

            print(
                f"🔄 Attempting Hugging Face config '{cfg}'..."
            )

            return load_dataset(
                dataset_name,
                name=cfg,
                token=token if token else None,
                trust_remote_code=True,
                verification_mode="no_checks"
            )

        except Exception as e:

            print(
                f"⚠️ Direct config load failed: {e}"
            )

            print(
                "🔄 Attempting automatic config detection..."
            )

    # --------------------------------------------------------------------------
    # Automatic config detection
    # --------------------------------------------------------------------------

    configs_to_try = []

    try:

        detected_configs = get_dataset_config_names(
            dataset_name,
            token=token if token else None
        )

        if detected_configs:

            print(
                f"ℹ️ Available dataset configs: "
                f"{detected_configs}"
            )

            configs_to_try = detected_configs

    except Exception:
        pass

    if not configs_to_try:
        configs_to_try = [None]

    # --------------------------------------------------------------------------
    # Try native Hugging Face loader
    # --------------------------------------------------------------------------

    last_error = None

    for c in configs_to_try:

        try:

            print(
                f"🔄 Trying Hugging Face config: {c}"
            )

            if c:

                return load_dataset(
                    dataset_name,
                    name=c,
                    token=token if token else None,
                    trust_remote_code=True,
                    verification_mode="no_checks"
                )

            else:

                return load_dataset(
                    dataset_name,
                    token=token if token else None,
                    trust_remote_code=True,
                    verification_mode="no_checks"
                )

        except Exception as e:

            last_error = e

            print(
                f"⚠️ Config '{c}' failed: {e}"
            )

    # ==========================================================================
    # ☁️ HUGGING FACE REPOSITORY FILE FALLBACK
    # ==========================================================================

    print(
        "🚀 Native HF dataset loader failed."
    )

    print(
        "🔄 Triggering Hugging Face repository-file fallback..."
    )

    try:

        repo_files = list_repo_files(
            dataset_name,
            repo_type="dataset",
            token=token if token else None
        )

        # ----------------------------------------------------------------------
        # Parquet
        # ----------------------------------------------------------------------

        train_parquet = [
            f
            for f in repo_files
            if (
                f.endswith(".parquet")
                and
                "train" in f.lower()
                and
                not any(
                    x in f.lower()
                    for x in [
                        "valid",
                        "eval",
                        "test",
                        "dev"
                    ]
                )
            )
        ]

        if not train_parquet:

            train_parquet = [
                f
                for f in repo_files
                if (
                    f.endswith(".parquet")
                    and
                    not any(
                        x in f.lower()
                        for x in [
                            "valid",
                            "eval",
                            "test",
                            "dev"
                        ]
                    )
                )
            ]

        if train_parquet:

            print(
                f"📦 Found {len(train_parquet)} "
                "Hugging Face train Parquet file(s)."
            )

            return load_dataset(
                "parquet",
                data_files={
                    "train": [
                        f"hf://datasets/{dataset_name}/{pf}"
                        for pf in train_parquet
                    ]
                }
            )

        # ----------------------------------------------------------------------
        # JSON / JSONL
        # ----------------------------------------------------------------------

        train_json = [
            f
            for f in repo_files
            if (
                f.endswith(
                    (
                        ".json",
                        ".jsonl",
                        ".jsonl.zst"
                    )
                )
                and
                "train" in f.lower()
                and
                not any(
                    x in f.lower()
                    for x in [
                        "valid",
                        "eval",
                        "test",
                        "dev"
                    ]
                )
            )
        ]

        if not train_json:

            train_json = [
                f
                for f in repo_files
                if (
                    f.endswith(
                        (
                            ".json",
                            ".jsonl",
                            ".jsonl.zst"
                        )
                    )
                    and
                    not any(
                        x in f.lower()
                        for x in [
                            "valid",
                            "eval",
                            "test",
                            "dev"
                        ]
                    )
                )
            ]

        if train_json:

            print(
                f"📦 Found {len(train_json)} "
                "Hugging Face train JSON file(s)."
            )

            return load_dataset(
                "json",
                data_files={
                    "train": [
                        f"hf://datasets/{dataset_name}/{jf}"
                        for jf in train_json
                    ]
                }
            )

    except Exception as fallback_error:

        print(
            f"⚠️ Hugging Face fallback failed: "
            f"{fallback_error}"
        )

    raise RuntimeError(
        f"❌ Failed to load dataset '{dataset_name}'.\n\n"
        f"Last error: {last_error}"
    )

raw_ds = bulletproof_load_dataset(DATASET_NAME, DATASET_CONFIG, HF_TOKEN)

split_name = None
if isinstance(raw_ds, dict) or hasattr(raw_ds, "keys"):
    available_splits = list(raw_ds.keys())
    print(f"ℹ️ Available dataset splits: {available_splits}")
    for preferred in ["train", "train_sft", "training", "train_eval", "default"]:
        if preferred in available_splits:
            split_name = preferred
            break
    if not split_name:
        split_name = available_splits[0]
    dataset = raw_ds[split_name]
else:
    dataset = raw_ds

print(f"✅ Selected Train split '{split_name}' with {len(dataset)} total samples.")

# Sample Slicing
try:
    max_samples_int = int(MAX_SAMPLES) if MAX_SAMPLES is not None and str(MAX_SAMPLES).strip() != "" else 0
except Exception:
    max_samples_int = 0

if max_samples_int > 0 and max_samples_int < len(dataset):
    dataset = dataset.select(range(max_samples_int))
    print(f"✂️ Sliced dataset to MAX_SAMPLES = {len(dataset)}")
else:
    print(f"ℹ️ Using FULL dataset ({len(dataset)} total samples).")

# --- STEP 6: MULTI-SAMPLE DATASET SCHEMA INSPECTOR & GUARD CHECKS ---
print("\n🔍 [6/10] Analyzing Dataset Modality Across 20 Samples...")

inspect_count = min(20, len(dataset))
sample_subset = [dataset[i] for i in range(inspect_count)]

all_keys = set()
key_counts = {}
for r in sample_subset:
    if isinstance(r, dict):
        for k in r.keys():
            all_keys.add(k)
            key_counts[k] = key_counts.get(k, 0) + 1

image_col = next((k for k in all_keys if k.lower() in ["image", "images", "img"]), None)
has_images = image_col is not None

# Guard 1: Multimodal on Text-Only Model
if has_images and not is_vision_model:
    raise ValueError(f"🛑 INCOMPATIBILITY DETECTED: Dataset contains images ('{image_col}' column), but base model '{MODEL_NAME}' is a Text-only LLM. Please select a Vision-Language model (e.g. Qwen/Qwen2-VL-2B-Instruct, meta-llama/Llama-3.2-11B-Vision-Instruct, llava-hf/llava-1.5-7b-hf).")

# Guard 2: DPO / Preference Guard
if "chosen" in all_keys and "rejected" in all_keys:
    sample_chosen = next((r.get("chosen") for r in sample_subset if r.get("chosen")), None)
    if isinstance(sample_chosen, (str, list, dict)):
        raise ValueError("🛑 UNSUPPORTED TASK: DPO / Preference dataset detected ('chosen' & 'rejected' columns). Use a DPO Trainer (DPOTrainer) instead of SFT.")

# Guard 3: Text Classification Guard
inst_candidates = ["instruction", "prompt", "question", "query", "problem", "input_text"]
answer_candidates = ["solution", "answer", "output", "response", "completion", "target", "output_text", "final_answer"]
has_gen_prompt = any(key_counts.get(k, 0) >= inspect_count * 0.5 for k in inst_candidates)
has_gen_target = any(key_counts.get(k, 0) >= inspect_count * 0.5 for k in answer_candidates)

if ("label" in all_keys or "labels" in all_keys) and not (has_gen_prompt and has_gen_target):
    lbl_k = "label" if "label" in all_keys else "labels"
    sample_lbl = next((r.get(lbl_k) for r in sample_subset if r.get(lbl_k) is not None), None)
    if isinstance(sample_lbl, (int, float)) or (isinstance(sample_lbl, list) and len(sample_lbl) > 0 and isinstance(sample_lbl[0], (int, float))):
        raise ValueError(f"🛑 UNSUPPORTED TASK: Text Classification / NLI dataset detected ('{lbl_k}' column with class IDs). Use AutoModelForSequenceClassification instead of Causal LM SFT.")

reasoning_candidates = ["reasoning", "thinking", "thought", "thoughts", "analysis", "reasoning_content", "chain_of_thought", "rationale", "explanation"]
context_candidates = ["input", "context"]

found_inst_col = next((k for k in inst_candidates if key_counts.get(k, 0) >= inspect_count * 0.5), None)
found_reasoning_col = next((k for k in reasoning_candidates if key_counts.get(k, 0) >= inspect_count * 0.5), None)
found_answer_col = next((k for k in answer_candidates if key_counts.get(k, 0) >= inspect_count * 0.5), None)
found_context_col = next((k for k in context_candidates if key_counts.get(k, 0) >= inspect_count * 0.5), None)

detected_format = "UNKNOWN"
detected_schema = ""

if has_images:
    if found_reasoning_col and found_answer_col:
        detected_format = "VLM_REASONING_SFT"
        detected_schema = f"{image_col} + {found_inst_col or 'prompt'} -> ({found_reasoning_col} + {found_answer_col})"
    else:
        detected_format = "VLM_SFT"
        detected_schema = f"{image_col} + {found_inst_col or 'prompt'} -> {found_answer_col or 'response'}"
elif key_counts.get("messages", 0) >= inspect_count * 0.5:
    detected_format, detected_schema = "CHAT_SFT", "messages"
elif key_counts.get("conversations", 0) >= inspect_count * 0.5:
    detected_format, detected_schema = "CHAT_SFT", "conversations"
elif found_inst_col and (found_reasoning_col or found_answer_col):
    if found_reasoning_col and found_answer_col:
        detected_format = "REASONING_TRACE_SFT"
        detected_schema = f"{found_inst_col} -> ({found_reasoning_col} + {found_answer_col})"
    elif found_reasoning_col:
        detected_format = "REASONING_ONLY_SFT"
        detected_schema = f"{found_inst_col} -> {found_reasoning_col}"
    else:
        detected_format = "INSTRUCTION_SFT"
        detected_schema = f"{found_inst_col} -> {found_answer_col}"
else:
    text_keys = ["text", "content", "body", "document", "raw", "code"]
    found_text = next((k for k in text_keys if key_counts.get(k, 0) >= inspect_count * 0.5), None)
    if found_text and len(all_keys) <= 3:
        detected_format, detected_schema = "PLAIN_TEXT_LM", found_text
    else:
        raise ValueError(
            f"🛑 AMBIGUOUS DATASET STRUCTURE DETECTED!\n"
            f"Dataset contains columns: {list(all_keys)}\n"
            f"No standard Chat ('messages', 'conversations'), Instruction ('{found_inst_col}'), or Plain Text column was recognized.\n"
            f"To prevent learning bad metadata, IDs, or raw tables, the trainer has safely stopped.\n"
            f"Please verify your dataset schema or format it to include standard 'messages', 'instruction' & 'output', or 'text' fields."
        )

# --- STEP 7: SHA-256 HASH LOCK & EXPERIMENT DIRECTORY ---
print("\n🔒 [7/10] Lock Hashing & Setting Up Persistent Storage Directory...")
num_epochs_val = float(NUM_EPOCHS) if NUM_EPOCHS and str(NUM_EPOCHS).strip() != "" else 1.0

# Pre-calculate steps
effective_batch_size = BATCH_SIZE * GRAD_ACCUM
steps_per_epoch = math.ceil(len(dataset) / effective_batch_size) if len(dataset) > 0 else 1
calculated_steps = math.ceil(steps_per_epoch * num_epochs_val)

try:
    user_max_steps = int(MAX_STEPS) if MAX_STEPS is not None and str(MAX_STEPS).strip() != "" else 0
except Exception:
    user_max_steps = 0

if user_max_steps > 0:
    TRAINING_MODE = "STEP_BASED"
    MAX_STEPS_RESOLVED = user_max_steps
else:
    TRAINING_MODE = "EPOCH_BASED"
    MAX_STEPS_RESOLVED = max(1, calculated_steps)

try:
    user_save_steps = int(SAVE_STEPS) if SAVE_STEPS is not None and str(SAVE_STEPS).strip() != "" else 0
except Exception:
    user_save_steps = 0

if user_save_steps > 0:
    SAVE_STEPS_RESOLVED = user_save_steps
else:
    SAVE_STEPS_RESOLVED = max(1, MAX_STEPS_RESOLVED // 3)

experiment_params = {
    "model_name": MODEL_NAME,
    "dataset_name": DATASET_NAME,
    "dataset_config": DATASET_CONFIG,
    "split_name": split_name,
    "learning_rate": LEARNING_RATE_RESOLVED,
    "batch_size": BATCH_SIZE,
    "grad_accum": GRAD_ACCUM,
    "num_epochs": num_epochs_val,
    "max_steps": MAX_STEPS_RESOLVED,
    "max_seq_len": MAX_SEQ_LENGTH_RESOLVED,
    "load_in_4bit": LOAD_IN_4BIT,
    "detected_format": detected_format,
    "model_route": model_route,
    "lora_r": 16,
    "lora_alpha": 16
}

canonical_config_str = json.dumps(experiment_params, sort_keys=True)
config_hash = hashlib.sha256(canonical_config_str.encode()).hexdigest()[:8]

folder_name = f"{MODEL_NAME.split('/')[-1]}__{DATASET_NAME.split('/')[-1]}__{config_hash}".replace(".", "_").replace("-", "_")
RUN_DIR = os.path.join(DRIVE_BASE_DIR, folder_name)
CHECKPOINTS_DIR = os.path.join(RUN_DIR, "checkpoints")
ADAPTER_DIR = os.path.join(RUN_DIR, "adapter")
MERGED_DIR = os.path.join(RUN_DIR, "merged_model")

os.makedirs(CHECKPOINTS_DIR, exist_ok=True)
os.makedirs(ADAPTER_DIR, exist_ok=True)
os.makedirs(MERGED_DIR, exist_ok=True)

print(f"📂 Persistent RUN Directory: {RUN_DIR}")
print(f"🔒 SHA-256 Experiment Hash Lock: {config_hash}\n")

with open(os.path.join(RUN_DIR, "run_config.json"), "w") as f:
    json.dump(experiment_params, f, indent=2)

# --- STEP 8: CAPABILITY-AWARE LOADER & VALIDATED LORA TARGETS ---
print("📥 [8/10] Loading Model & Processor/Tokenizer based on Capability Route...")
use_unsloth = False
model = None
tokenizer = None
processor = None

def get_validated_lora_targets(model_obj):
    import torch.nn as nn
    lora_module_names = set()
    ignore_keywords = ["lm_head", "embed_tokens", "embed", "wte", "wpe", "score", "classifier", "head", "output_layer", "vision_model", "vision_tower", "patch_embedding"]

    for name, module in model_obj.named_modules():
        if isinstance(module, nn.Linear) or "Linear" in module.__class__.__name__ or "Conv1D" in module.__class__.__name__:
            names = name.split('.')
            layer_basename = names[-1]
            if not any(ig in name.lower() for ig in ignore_keywords):
                lora_module_names.add(layer_basename)

    target_list = sorted(list(lora_module_names))
    if not target_list:
        return "all-linear"
    return target_list

if model_route == "VISION_LANGUAGE":
    print("👁️ Vision-Language Model Branch Loading...")
    try:
        print("🚀 Attempting loading with Unsloth FastVisionModel...")
        from unsloth import FastVisionModel
        model, tokenizer = FastVisionModel.from_pretrained(
            model_name = MODEL_NAME,
            load_in_4bit = LOAD_IN_4BIT,
            token = HF_TOKEN if HF_TOKEN else None,
        )
        detected_targets = get_validated_lora_targets(model)
        model = FastVisionModel.get_peft_model(
            model,
            r = 16,
            target_modules = detected_targets,
            lora_alpha = 16,
            use_gradient_checkpointing = "unsloth",
            random_state = 3407,
        )
        use_unsloth = True
        print("✅ Unsloth FastVisionModel initialized successfully!\n")
    except Exception as e_v:
        print(f"⚠️ Unsloth FastVisionModel skipped ({e_v}). Switching to Transformers AutoProcessor + Vision Model...")
        from transformers import AutoProcessor, AutoModelForVision2Seq, AutoModelForConditionalGeneration, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
        from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

        quant_config = None
        if LOAD_IN_4BIT:
            quant_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.bfloat16 if has_bf16 else torch.float16,
                bnb_4bit_use_double_quant=True
            )

        try:
            processor = AutoProcessor.from_pretrained(MODEL_NAME, token=HF_TOKEN if HF_TOKEN else None, trust_remote_code=True)
        except Exception:
            processor = None

        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN if HF_TOKEN else None, trust_remote_code=True)
        if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

        try:
            model = AutoModelForVision2Seq.from_pretrained(MODEL_NAME, quantization_config=quant_config, device_map="auto", token=HF_TOKEN if HF_TOKEN else None, trust_remote_code=True)
        except Exception:
            try:
                model = AutoModelForConditionalGeneration.from_pretrained(MODEL_NAME, quantization_config=quant_config, device_map="auto", token=HF_TOKEN if HF_TOKEN else None, trust_remote_code=True)
            except Exception:
                model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=quant_config, device_map="auto", token=HF_TOKEN if HF_TOKEN else None, trust_remote_code=True)

        model = prepare_model_for_kbit_training(model)
        detected_targets = get_validated_lora_targets(model)
        peft_config = LoraConfig(r=16, lora_alpha=16, target_modules=detected_targets, lora_dropout=0.05, bias="none")
        model = get_peft_model(model, peft_config)
        print("✅ Transformers VLM + PEFT initialized successfully!\n")

elif model_route == "SEQ2SEQ_LM":
    print("🔄 Seq2Seq / Encoder-Decoder Branch Loading...")
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

    quant_config = None
    if LOAD_IN_4BIT:
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16 if has_bf16 else torch.float16,
            bnb_4bit_use_double_quant=True
        )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN if HF_TOKEN else None, trust_remote_code=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, quantization_config=quant_config, device_map="auto", token=HF_TOKEN if HF_TOKEN else None, trust_remote_code=True)
    model = prepare_model_for_kbit_training(model)
    detected_targets = get_validated_lora_targets(model)
    peft_config = LoraConfig(r=16, lora_alpha=16, target_modules=detected_targets, lora_dropout=0.05, bias="none", task_type="SEQ_2_SEQ_LM")
    model = get_peft_model(model, peft_config)
    print("✅ Transformers Seq2SeqLM + PEFT initialized successfully!\n")

else:
    print("🤖 Causal Language Model Branch Loading...")
    try:
        print("🚀 Attempting loading with Unsloth FastLanguageModel...")
        from unsloth import FastLanguageModel
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name = MODEL_NAME,
            max_seq_length = MAX_SEQ_LENGTH_RESOLVED,
            dtype = None,
            load_in_4bit = LOAD_IN_4BIT,
            token = HF_TOKEN if HF_TOKEN else None,
        )
        detected_targets = get_validated_lora_targets(model)
        model = FastLanguageModel.get_peft_model(
            model,
            r = 16,
            target_modules = detected_targets,
            lora_alpha = 16,
            lora_dropout = 0,
            bias = "none",
            use_gradient_checkpointing = "unsloth",
            random_state = 3407,
        )
        use_unsloth = True
        print("✅ Unsloth FastLanguageModel initialized successfully!\n")
    except Exception as e_t:
        print(f"⚠️ Unsloth FastLanguageModel skipped ({e_t}). Switching to Transformers AutoModelForCausalLM...")
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
        from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

        quant_config = None
        if LOAD_IN_4BIT:
            quant_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.bfloat16 if has_bf16 else torch.float16,
                bnb_4bit_use_double_quant=True
            )

        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN if HF_TOKEN else None, trust_remote_code=True)
        if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

        model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=quant_config, device_map="auto", token=HF_TOKEN if HF_TOKEN else None, trust_remote_code=True)
        model = prepare_model_for_kbit_training(model)
        detected_targets = get_validated_lora_targets(model)
        peft_config = LoraConfig(r=16, lora_alpha=16, target_modules=detected_targets, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM")
        model = get_peft_model(model, peft_config)
        print("✅ Transformers CausalLM + PEFT initialized successfully!\n")

# Format row strict
actual_format_name = "Model-Aware Template"

def format_chat_messages_model_aware(messages, model_name, tokenizer_obj):
    if tokenizer_obj and hasattr(tokenizer_obj, "apply_chat_template") and getattr(tokenizer_obj, "chat_template", None) is not None:
        try:
            return tokenizer_obj.apply_chat_template(messages, tokenize=False, add_generation_prompt=False), "Native Tokenizer Chat Template"
        except Exception:
            pass

    m_lower = model_name.lower()
    if "gemma" in m_lower:
        parts = []
        for m in messages:
            if isinstance(m, dict):
                r = 'user' if m.get('role') in ['user','human'] else 'model'
                c = m.get('content', '')
                parts.append(f"<start_of_turn>{r}\n{c}<end_of_turn>")
        return "\n".join(parts).strip(), "Gemma Family Template"
    if any(x in m_lower for x in ["llama", "mistral", "mixtral"]):
        user_msg = next((m.get("content","") for m in messages if isinstance(m, dict) and m.get("role") in ["user", "human"]), "")
        assistant_msg = next((m.get("content","") for m in messages if isinstance(m, dict) and m.get("role") in ["assistant", "gpt"]), "")
        return f"[INST] {user_msg} [/INST] {assistant_msg}".strip(), "Llama / Mistral Family Template"
    if "phi" in m_lower:
        parts = []
        for m in messages:
            if isinstance(m, dict):
                r = 'user' if m.get('role') in ['user','human'] else 'assistant'
                c = m.get('content', '')
                parts.append(f"<|{r}|>\n{c}<|end|>")
        return "\n".join(parts).strip(), "Phi Family Template"
    if "qwen" in m_lower:
        parts = []
        for m in messages:
            if isinstance(m, dict):
                r = 'user' if m.get('role') in ['user','human'] else 'assistant'
                c = m.get('content', '')
                parts.append(f"<|im_start|>{r}\n{c}<|im_end|>")
        return "\n".join(parts).strip(), "Qwen ChatML Template"

    parts = []
    for m in messages:
        if isinstance(m, dict):
            r = 'User' if m.get('role') in ['user','human'] else 'Assistant'
            c = m.get('content', '')
            parts.append(f"{r}: {c}")
    return "\n".join(parts).strip(), "Clean Standard SFT Fallback"

def format_row_strict(row):
    global actual_format_name
    if has_images and image_col in row:
        actual_format_name = "VLM AutoProcessor Pipeline"
        inst_val = str(row.get(found_inst_col or "prompt", "")).strip()
        reasoning_val = str(row.get(found_reasoning_col, "")).strip() if found_reasoning_col and found_reasoning_col in row else ""
        answer_val = str(row.get(found_answer_col or "output", "")).strip() if found_answer_col and found_answer_col in row else ""

        if reasoning_val and answer_val:
            assistant_content = f"<think>\n{reasoning_val}\n</think>\n{answer_val}"
        else:
            assistant_content = answer_val or reasoning_val

        msgs = [{"role": "user", "content": inst_val}, {"role": "assistant", "content": assistant_content}]
        formatted_text, _ = format_chat_messages_model_aware(msgs, MODEL_NAME, tokenizer)
        res = dict(row)
        res["text"] = formatted_text
        return res

    if "messages" in row and isinstance(row["messages"], list):
        formatted_text, actual_format_name = format_chat_messages_model_aware(row["messages"], MODEL_NAME, tokenizer)
        return {"text": formatted_text}

    if "conversations" in row and isinstance(row["conversations"], list):
        norm_msgs = []
        for m in row["conversations"]:
            if isinstance(m, dict):
                r = m.get("from", m.get("role", "user"))
                c = m.get("value", m.get("content", ""))
                r = "user" if r in ["human", "user"] else "assistant"
                norm_msgs.append({"role": r, "content": c})
        formatted_text, actual_format_name = format_chat_messages_model_aware(norm_msgs, MODEL_NAME, tokenizer)
        return {"text": formatted_text}

    if found_inst_col and found_inst_col in row:
        inst_val = str(row.get(found_inst_col, "")).strip()
        context_val = str(row.get(found_context_col, "")).strip() if found_context_col and found_context_col in row else ""
        full_user = f"{inst_val}\nContext: {context_val}" if context_val else inst_val

        reasoning_val = str(row.get(found_reasoning_col, "")).strip() if found_reasoning_col and found_reasoning_col in row else ""
        answer_val = str(row.get(found_answer_col, "")).strip() if found_answer_col and found_answer_col in row else ""

        if reasoning_val and answer_val:
            assistant_content = f"<think>\n{reasoning_val}\n</think>\n{answer_val}"
        elif answer_val:
            assistant_content = answer_val
        elif reasoning_val:
            assistant_content = f"<think>\n{reasoning_val}\n</think>"
        else:
            return {"text": ""}

        msgs = [{"role": "user", "content": full_user}, {"role": "assistant", "content": assistant_content}]
        formatted_text, actual_format_name = format_chat_messages_model_aware(msgs, MODEL_NAME, tokenizer)
        return {"text": formatted_text}

    text_keys = ["text", "content", "body", "document", "raw", "code"]
    found_text = next((k for k in text_keys if k in row and row[k]), None)
    if found_text:
        actual_format_name = "Raw Text Column"
        return {"text": str(row[found_text]).strip()}

    return {"text": ""}

num_map_proc = 1 if (has_images or is_vision_model) else 2
formatted_dataset = dataset.map(format_row_strict, batched=False, num_proc=num_map_proc)
formatted_dataset = formatted_dataset.filter(lambda x: len(x.get("text", "").strip()) > 0)
print(f"✅ Dataset formatted ({detected_format})! {len(formatted_dataset)} valid rows ready.\n")

# --- STEP 9: PRE-TRAINING DIAGNOSTIC SUMMARY DASHBOARD ---
print("=" * 65)
print("📋 PRE-TRAINING VALIDATION SUMMARY DASHBOARD")
print("=" * 65)
print(f"🤖 Base Model:           {MODEL_NAME}")
print(f"🎯 Model Route:          {model_route}")
print(f"🎯 Auto LoRA Targets:   {detected_targets}")
print(f"📊 Dataset Name:         {DATASET_NAME} ({len(formatted_dataset)} samples)")
print(f"🏷️  Detected Task:        {detected_format} ({detected_schema})")
print(f"🎨 Chat Formatter:       {actual_format_name}")
print(f"📏 Max Sequence Length:  {MAX_SEQ_LENGTH_RESOLVED if MAX_SEQ_LENGTH_RESOLVED is not None else 'None (VLM Preserved)'}")
print(f"⚡ Hardware Tuning:      Batch Size = {BATCH_SIZE}, Grad Accum = {GRAD_ACCUM} (Effective = {effective_batch_size})")
print(f"📈 Training Target:      Mode = {TRAINING_MODE} ({num_epochs_val} Epochs) -> {MAX_STEPS_RESOLVED} Total Steps")
print(f"💾 Checkpoint Frequency: Every {SAVE_STEPS_RESOLVED} steps")
print(f"📂 Persistent Run Path:  {RUN_DIR}")
print("-" * 65)
print("🔍 SAMPLE FORMATTED PROMPT PREVIEW (Sample #1):")
print("-" * 65)
sample_preview = formatted_dataset[0]["text"]
preview_text = sample_preview[:600] + ("\n... [Truncated for preview]" if len(sample_preview) > 600 else "")
print(preview_text)
print("=" * 65 + "\n")

# --- STEP 10: INITIALIZE TRAINER & VALIDATED CHECKPOINT RESUME ---
print("🛠️ [9/10] Initializing SFTTrainer & Checking Validated Resume State...")
from trl import SFTTrainer
from transformers import TrainingArguments

def validate_checkpoint_dir(checkpoint_path):
    if not os.path.exists(checkpoint_path) or not os.path.isdir(checkpoint_path):
        return False
    files = os.listdir(checkpoint_path)
    valid_indicators = ["adapter_model.safetensors", "adapter_model.bin", "pytorch_model.bin", "model.safetensors", "trainer_state.json"]
    return any(f in files for f in valid_indicators)

resume_from_checkpoint = None
if os.path.exists(CHECKPOINTS_DIR):
    candidate_cps = [d for d in os.listdir(CHECKPOINTS_DIR) if d.startswith("checkpoint-")]
    if candidate_cps:
        cps_sorted = sorted(candidate_cps, key=lambda x: int(x.split("-")[-1]) if x.split("-")[-1].isdigit() else 0, reverse=True)
        for cand in cps_sorted:
            cand_path = os.path.join(CHECKPOINTS_DIR, cand)
            if validate_checkpoint_dir(cand_path):
                resume_from_checkpoint = cand_path
                print(f"🔄 AUTO-RESUMING from validated checkpoint: {cand}")
                break

if not resume_from_checkpoint:
    print("🚀 STARTING FRESH TRAINING RUN...")

trainer_kwargs = {
    "model": model,
    "train_dataset": formatted_dataset,
    "dataset_text_field": "text",
    "max_seq_length": MAX_SEQ_LENGTH_RESOLVED,
    "dataset_num_proc": 1 if (has_images or is_vision_model) else 2,
    "packing": False,
    "args": TrainingArguments(
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_steps = 5,
        max_steps = MAX_STEPS_RESOLVED,
        learning_rate = LEARNING_RATE_RESOLVED,
        fp16 = not has_bf16,
        bf16 = has_bf16,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = CHECKPOINTS_DIR,
        save_strategy = "steps",
        save_steps = SAVE_STEPS_RESOLVED,
        save_total_limit = 3,
        remove_unused_columns = False if (has_images or is_vision_model) else True,
    ),
}

if processor is not None:
    trainer_kwargs["processor"] = processor
elif tokenizer is not None:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = SFTTrainer(**trainer_kwargs)

# Launch Training
print("🔥 [10/10] Training in progress...")
trainer.train(resume_from_checkpoint=resume_from_checkpoint)
print("🎉 Training Completed Successfully!\n")

# --- FAULT-TOLERANT EXPORT PIPELINE ---
EXPORT_MERGED = "/content/merged_16bit_model"
EXPORT_ADAPTER = "/content/adapter_model"

print("📦 Exporting Fine-Tuned Model Outputs...")

# 1. Always save PEFT LoRA Adapter first!
try:
    print(f"💾 [1/2] Saving PEFT LoRA Adapter to '{ADAPTER_DIR}'...")
    model.save_pretrained(ADAPTER_DIR)
    if tokenizer: tokenizer.save_pretrained(ADAPTER_DIR)
    if processor: processor.save_pretrained(ADAPTER_DIR)

    # Symlink or copy to /content/adapter_model
    if os.path.exists(EXPORT_ADAPTER): shutil.rmtree(EXPORT_ADAPTER)
    shutil.copytree(ADAPTER_DIR, EXPORT_ADAPTER)
    print("✅ PEFT LoRA Adapter saved successfully!")
except Exception as e_ad:
    print(f"⚠️ Adapter save warning ({e_ad}).")

# 2. Attempt 16-bit Full Model Merge
try:
    print(f"📦 [2/2] Merging 16-bit Full Model to '{MERGED_DIR}'...")
    if use_unsloth:
        model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method="merged_16bit")
    else:
        merged_model = model.merge_and_unload()
        merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
        if tokenizer: tokenizer.save_pretrained(MERGED_DIR)
        if processor: processor.save_pretrained(MERGED_DIR)

    if os.path.exists(EXPORT_MERGED): shutil.rmtree(EXPORT_MERGED)
    shutil.copytree(MERGED_DIR, EXPORT_MERGED)
    print("✅ 16-bit Merged Base Model exported successfully!")
except Exception as e_merge:
    print(f"⚠️ 16-bit Full Model Merging skipped ({e_merge}).")
    print(f"💡 Your PEFT LoRA Adapter is safely preserved at '{ADAPTER_DIR}' and '{EXPORT_ADAPTER}'.")

status_data = {
    "status": "COMPLETED",
    "run_dir": RUN_DIR,
    "adapter_dir": ADAPTER_DIR,
    "merged_dir": MERGED_DIR
}
with open(os.path.join(RUN_DIR, "training_status.json"), "w") as f:
    json.dump(status_data, f, indent=2)

print("\n🏆 UNIVERSAL SMART FINE-TUNING PIPELINE FINISHED SUCCESSFULLY!")


### 🛑 Want to stop training and keep your progress?

You can safely stop **Cell 1** at any point during training.
Then run **Cell 2** to create a **partially merged AI model** using the latest saved training checkpoint.

> ⚠️ Only the progress saved in the latest checkpoint can be recovered. Any training progress after that checkpoint may be lost.


In [ ]:
# @title 🔄 Recover & Create Partially Merged Model
# ==============================================================================
# 🚀 UNIVERSAL CHECKPOINT RECOVERY + LoRA MERGE
#
# Finds the exact MODEL + DATASET training run in Google Drive,
# selects the latest usable checkpoint, merges it into the ORIGINAL
# base model, and saves the FULL merged model locally in Colab.
#
# DEFAULT OUTPUT:
#     /content/Partially Merged Model
#
# IMPORTANT:
#   ✅ Searches Google Drive
#   ✅ Requires exact Model + Dataset match
#   ✅ Uses latest valid checkpoint
#   ✅ Merges LoRA into base model
#   ✅ Saves merged model to /content only
#   ✅ Saves tokenizer
#   ✅ No GGUF
#   ✅ Does NOT modify/delete the training checkpoints in Drive
# ==============================================================================


# ==============================================================================
# 🎛️ USER INPUT
# ==============================================================================

RECOVER_MODEL_NAME = "Qwen/Qwen3-1.7B"                    #@param {type:"string"}
RECOVER_DATASET_NAME = "bespokelabs/Bespoke-Stratos-17k" #@param {type:"string"}

# Default = LOCAL COLAB STORAGE
RECOVER_OUTPUT_PATH = "/content/Partially-Merged-Model"  #@param {type:"string"}

# Your training code's Google Drive root
DRIVE_BASE_DIR = "/content/drive/MyDrive/unsloth_checkpoints"

# --- STEP 1: MOUNT GOOGLE DRIVE & PERSISTENT STORAGE HIERARCHY ---

DRIVE_BASE_DIR = "/content/drive/MyDrive/unsloth_checkpoints"
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print("✅ Google Drive connected successfully!")
except Exception as e:
    print(f"⚠️ Google Drive not mounted ({e}). Using persistent local storage /content/checkpoints")
    DRIVE_BASE_DIR = "/content/checkpoints"


# ==============================================================================
# STEP 1 — IMPORT BASIC LIBRARIES ONLY
# ==============================================================================

import os
import json
import shutil
import torch


# ==============================================================================
# STEP 2 — INPUT VALIDATION
# ==============================================================================

print("=" * 80)
print("🔄 PARTIALLY MERGED MODEL RECOVERY")
print("=" * 80)
print()

RECOVER_MODEL_NAME = str(RECOVER_MODEL_NAME).strip()
RECOVER_DATASET_NAME = str(RECOVER_DATASET_NAME).strip()
RECOVER_OUTPUT_PATH = str(RECOVER_OUTPUT_PATH).strip()

if not RECOVER_MODEL_NAME:
    raise ValueError("❌ Model name cannot be empty.")

if not RECOVER_DATASET_NAME:
    raise ValueError("❌ Dataset name cannot be empty.")

if not RECOVER_OUTPUT_PATH:
    RECOVER_OUTPUT_PATH = "/content/Partially Merged Model"

RECOVER_OUTPUT_PATH = os.path.abspath(RECOVER_OUTPUT_PATH)


# Safety: this tool must save locally, not to Google Drive
if RECOVER_OUTPUT_PATH.startswith("/content/drive"):
    raise ValueError(
        "🛑 INVALID OUTPUT PATH\n\n"
        "The merged model must be saved outside Google Drive.\n\n"
        "Recommended:\n"
        "/content/Partially Merged Model"
    )


print(f"🤖 Model        : {RECOVER_MODEL_NAME}")
print(f"📊 Dataset      : {RECOVER_DATASET_NAME}")
print(f"💾 Output       : {RECOVER_OUTPUT_PATH}")
print(f"☁️ Search Drive : {DRIVE_BASE_DIR}")
print()


# ==============================================================================
# STEP 3 — CHECK TRAINING STORAGE
# ==============================================================================

if not os.path.isdir(DRIVE_BASE_DIR):

    print("=" * 80)
    print("❌ TRAINING STORAGE NOT FOUND")
    print("=" * 80)
    print()
    print(f"Expected:")
    print(f"  {DRIVE_BASE_DIR}")
    print()
    print("⚠️ Recoverable trained samples: 0")
    print("❌ Cannot create a merged model.")
    raise SystemExit


print("✅ Google Drive training storage found.")
print()


# ==============================================================================
# STEP 4 — EXACT MODEL + DATASET SEARCH
# ==============================================================================

print("=" * 80)
print("🔎 SEARCHING FOR MATCHING MODEL + DATASET")
print("=" * 80)
print()

matching_runs = []


for folder_name in os.listdir(DRIVE_BASE_DIR):

    run_dir = os.path.join(
        DRIVE_BASE_DIR,
        folder_name
    )

    if not os.path.isdir(run_dir):
        continue

    config_path = os.path.join(
        run_dir,
        "run_config.json"
    )

    if not os.path.isfile(config_path):
        continue

    try:

        with open(
            config_path,
            "r",
            encoding="utf-8"
        ) as f:
            config = json.load(f)

    except Exception:
        continue

    saved_model = str(
        config.get("model_name", "")
    ).strip()

    saved_dataset = str(
        config.get("dataset_name", "")
    ).strip()

    # EXACT match
    if (
        saved_model == RECOVER_MODEL_NAME
        and
        saved_dataset == RECOVER_DATASET_NAME
    ):

        matching_runs.append({
            "run_dir": run_dir,
            "config": config
        })


# ==============================================================================
# STEP 5 — NO EXACT MATCH
# ==============================================================================

if not matching_runs:

    print("=" * 80)
    print("❌ NO MATCHING TRAINING RUN FOUND")
    print("=" * 80)
    print()
    print(f"Requested Model   : {RECOVER_MODEL_NAME}")
    print(f"Requested Dataset : {RECOVER_DATASET_NAME}")
    print()
    print("⚠️ Model + Dataset did not exactly match any saved run.")
    print("⚠️ Recoverable trained samples: 0")
    print("❌ A partially merged model cannot be generated.")
    raise SystemExit


print(f"✅ Matching run(s) found: {len(matching_runs)}")
print()


# ==============================================================================
# STEP 6 — FIND ALL VALID CHECKPOINTS
# ==============================================================================

print("=" * 80)
print("🧪 INSPECTING CHECKPOINTS")
print("=" * 80)
print()

candidates = []


def get_checkpoint_step(path):

    name = os.path.basename(path)

    if not name.startswith("checkpoint-"):
        return -1

    try:
        return int(name.rsplit("-", 1)[-1])
    except Exception:
        return -1


def has_adapter_weights(directory):

    if not os.path.isdir(directory):
        return False

    safetensors_path = os.path.join(
        directory,
        "adapter_model.safetensors"
    )

    bin_path = os.path.join(
        directory,
        "adapter_model.bin"
    )

    return (
        os.path.isfile(safetensors_path)
        or
        os.path.isfile(bin_path)
    )


for run in matching_runs:

    run_dir = run["run_dir"]
    config = run["config"]

    checkpoints_dir = os.path.join(
        run_dir,
        "checkpoints"
    )

    adapter_dir = os.path.join(
        run_dir,
        "adapter"
    )

    # --------------------------------------------------------------------------
    # Check checkpoint-XXXX directories
    # --------------------------------------------------------------------------

    if os.path.isdir(checkpoints_dir):

        for item in os.listdir(checkpoints_dir):

            checkpoint_dir = os.path.join(
                checkpoints_dir,
                item
            )

            if not os.path.isdir(checkpoint_dir):
                continue

            if not item.startswith("checkpoint-"):
                continue

            step = get_checkpoint_step(
                checkpoint_dir
            )

            if step < 0:
                continue

            if has_adapter_weights(
                checkpoint_dir
            ):

                candidates.append({
                    "run_dir": run_dir,
                    "config": config,
                    "source": checkpoint_dir,
                    "source_type": "checkpoint",
                    "step": step
                })


    # --------------------------------------------------------------------------
    # Check final adapter
    # --------------------------------------------------------------------------

    if has_adapter_weights(adapter_dir):

        candidates.append({
            "run_dir": run_dir,
            "config": config,
            "source": adapter_dir,
            "source_type": "adapter",
            "step": -1
        })


# ==============================================================================
# STEP 7 — NO USABLE WEIGHTS
# ==============================================================================

if not candidates:

    print("=" * 80)
    print("❌ MATCH FOUND — BUT NO USABLE TRAINED WEIGHTS FOUND")
    print("=" * 80)
    print()
    print("The model and dataset matched, but there is no usable")
    print("LoRA checkpoint or adapter.")
    print()
    print("⚠️ Recoverable trained samples: 0")
    print("❌ A partially merged model cannot be generated.")
    raise SystemExit


# ==============================================================================
# STEP 8 — SELECT LATEST CHECKPOINT
# ==============================================================================

best_candidate = max(
    candidates,
    key=lambda x: x["step"]
)

RUN_DIR = best_candidate["run_dir"]
CONFIG = best_candidate["config"]
SOURCE_PATH = best_candidate["source"]
SOURCE_TYPE = best_candidate["source_type"]

CHECKPOINT_STEP = best_candidate["step"]


# ==============================================================================
# STEP 9 — SHOW WHAT WILL BE RECOVERED
# ==============================================================================

print("=" * 80)
print("✅ RECOVERABLE TRAINING FOUND")
print("=" * 80)
print()

print(f"🤖 Base Model       : {RECOVER_MODEL_NAME}")
print(f"📊 Dataset          : {RECOVER_DATASET_NAME}")

print(
    f"🧠 Model Route      : "
    f"{CONFIG.get('model_route', 'CAUSAL_LM')}"
)

print(f"📂 Training Run     : {RUN_DIR}")
print(f"💾 Selected Weights : {SOURCE_PATH}")
print(f"🔧 Source Type      : {SOURCE_TYPE}")

if CHECKPOINT_STEP >= 0:
    print(f"📈 Checkpoint Step  : {CHECKPOINT_STEP}")

print()


# ==============================================================================
# STEP 10 — READ TRAINER STATE
# ==============================================================================

TRAINED_GLOBAL_STEP = CHECKPOINT_STEP
TRAINED_EPOCH = None

if SOURCE_TYPE == "checkpoint":

    trainer_state_path = os.path.join(
        SOURCE_PATH,
        "trainer_state.json"
    )

    if os.path.isfile(trainer_state_path):

        try:

            with open(
                trainer_state_path,
                "r",
                encoding="utf-8"
            ) as f:
                trainer_state = json.load(f)

            TRAINED_GLOBAL_STEP = int(
                trainer_state.get(
                    "global_step",
                    CHECKPOINT_STEP
                )
            )

            if trainer_state.get("epoch") is not None:

                TRAINED_EPOCH = float(
                    trainer_state["epoch"]
                )

        except Exception:
            TRAINED_GLOBAL_STEP = CHECKPOINT_STEP


print(
    f"📊 Recoverable Training Steps : "
    f"{TRAINED_GLOBAL_STEP}"
)

if TRAINED_EPOCH is not None:

    print(
        f"📚 Recoverable Epoch Progress  : "
        f"{TRAINED_EPOCH:.4f}"
    )

print()


# ==============================================================================
# STEP 11 — ESTIMATE REPRESENTED SAMPLES
# ==============================================================================

try:

    batch_size = int(
        CONFIG.get("batch_size", 1)
    )

    grad_accum = int(
        CONFIG.get("grad_accum", 1)
    )

    effective_batch_size = max(
        1,
        batch_size * grad_accum
    )

    estimated_samples = (
        TRAINED_GLOBAL_STEP
        * effective_batch_size
    )

    print(
        f"📦 Approx. Samples Represented : "
        f"{estimated_samples}"
    )

except Exception:

    effective_batch_size = None
    estimated_samples = None

    print(
        "📦 Approx. Samples Represented : "
        "Unable to calculate"
    )

print()


# ==============================================================================
# STEP 12 — LOAD ONLY REQUIRED TRANSFORMERS CLASSES
# ==============================================================================

print("=" * 80)
print("📦 PREPARING MODEL LOADER")
print("=" * 80)
print()


# IMPORTANT:
# Do NOT import AutoModelForConditionalGeneration.
#
# That class caused the error in your previous run.
#
# We only import the model class actually needed.

from transformers import AutoTokenizer
from peft import PeftModel


# ==============================================================================
# STEP 13 — DETERMINE MODEL ROUTE
# ==============================================================================

MODEL_ROUTE = str(
    CONFIG.get(
        "model_route",
        "CAUSAL_LM"
    )
).upper()


print(f"🧠 Saved Model Route : {MODEL_ROUTE}")
print()


# ==============================================================================
# STEP 14 — LOAD TOKENIZER
# ==============================================================================

print("📥 Loading tokenizer...")


tokenizer = AutoTokenizer.from_pretrained(
    RECOVER_MODEL_NAME,
    trust_remote_code=True
)


if tokenizer.pad_token is None:

    tokenizer.pad_token = tokenizer.eos_token


print("✅ Tokenizer loaded.")
print()


# ==============================================================================
# STEP 15 — LOAD BASE MODEL
# ==============================================================================

print("=" * 80)
print("📥 LOADING ORIGINAL BASE MODEL")
print("=" * 80)
print()

print(
    "⚠️ Loading base model in FP16 for LoRA merging."
)

print(
    "⚠️ The base model is NOT loaded in 4-bit for this merge."
)

print()


MERGE_DTYPE = torch.float16


# ------------------------------------------------------------------------------
# CAUSAL LM
# ------------------------------------------------------------------------------

if MODEL_ROUTE == "CAUSAL_LM":

    from transformers import AutoModelForCausalLM

    print("🤖 Loading Causal Language Model...")

    base_model = AutoModelForCausalLM.from_pretrained(
        RECOVER_MODEL_NAME,
        torch_dtype=MERGE_DTYPE,
        device_map="auto",
        low_cpu_mem_usage=True,
        trust_remote_code=True
    )


# ------------------------------------------------------------------------------
# SEQ2SEQ
# ------------------------------------------------------------------------------

elif MODEL_ROUTE == "SEQ2SEQ_LM":

    from transformers import AutoModelForSeq2SeqLM

    print("🔄 Loading Seq2Seq model...")

    base_model = AutoModelForSeq2SeqLM.from_pretrained(
        RECOVER_MODEL_NAME,
        torch_dtype=MERGE_DTYPE,
        device_map="auto",
        low_cpu_mem_usage=True,
        trust_remote_code=True
    )


# ------------------------------------------------------------------------------
# VISION LANGUAGE
# ------------------------------------------------------------------------------

elif MODEL_ROUTE == "VISION_LANGUAGE":

    # Import only inside this branch.
    #
    # This prevents a VLM-only class from breaking normal
    # text-model recovery.

    try:

        from transformers import AutoModelForVision2Seq

        print(
            "👁️ Loading Vision-Language model..."
        )

        base_model = AutoModelForVision2Seq.from_pretrained(
            RECOVER_MODEL_NAME,
            torch_dtype=MERGE_DTYPE,
            device_map="auto",
            low_cpu_mem_usage=True,
            trust_remote_code=True
        )

    except ImportError as e:

        raise RuntimeError(
            "❌ This installed Transformers version does not "
            "provide AutoModelForVision2Seq.\n\n"
            "Your current model is marked as VISION_LANGUAGE, "
            "so the appropriate VLM loader is required.\n\n"
            f"Original error:\n{e}"
        )

    except Exception as e:

        raise RuntimeError(
            "❌ Failed to load the Vision-Language base model.\n\n"
            f"Error:\n{e}"
        )


# ------------------------------------------------------------------------------
# UNKNOWN ROUTE
# ------------------------------------------------------------------------------

else:

    print(
        "⚠️ Unknown saved route."
    )

    print(
        "🔄 Falling back to AutoModelForCausalLM."
    )

    from transformers import AutoModelForCausalLM

    base_model = AutoModelForCausalLM.from_pretrained(
        RECOVER_MODEL_NAME,
        torch_dtype=MERGE_DTYPE,
        device_map="auto",
        low_cpu_mem_usage=True,
        trust_remote_code=True
    )


print()
print("✅ Base model loaded.")
print()


# ==============================================================================
# STEP 16 — LOAD TRAINED LoRA
# ==============================================================================

print("=" * 80)
print("🔗 LOADING TRAINED LoRA ADAPTER")
print("=" * 80)
print()

print(
    f"📂 Adapter source:\n"
    f"   {SOURCE_PATH}"
)

print()


try:

    trained_model = PeftModel.from_pretrained(
        base_model,
        SOURCE_PATH,
        is_trainable=False
    )

except Exception as e:

    raise RuntimeError(
        "❌ FAILED TO LOAD LoRA ADAPTER.\n\n"
        f"Adapter:\n{SOURCE_PATH}\n\n"
        f"Error:\n{e}"
    )


print("✅ Trained LoRA adapter loaded.")
print()


# ==============================================================================
# STEP 17 — MERGE LoRA
# ==============================================================================

print("=" * 80)
print("🔄 MERGING LoRA → BASE MODEL")
print("=" * 80)
print()

print(
    "⏳ Creating the merged full model..."
)

print(
    "⏳ Please do not interrupt this step."
)

print()


try:

    merged_model = trained_model.merge_and_unload()

except Exception as e:

    raise RuntimeError(
        "❌ LoRA MERGE FAILED.\n\n"
        f"Error:\n{e}"
    )


print()
print("✅ LoRA successfully merged.")
print()


# ==============================================================================
# STEP 18 — PREPARE LOCAL OUTPUT
# ==============================================================================

print("=" * 80)
print("💾 PREPARING LOCAL COLAB OUTPUT")
print("=" * 80)
print()


# We only remove the exact output folder.
# We NEVER touch /content itself.
# We NEVER touch Google Drive checkpoints.

if os.path.exists(RECOVER_OUTPUT_PATH):

    print(
        "⚠️ Existing output directory detected."
    )

    shutil.rmtree(
        RECOVER_OUTPUT_PATH
    )

    print(
        "✅ Existing output directory replaced."
    )


os.makedirs(
    RECOVER_OUTPUT_PATH,
    exist_ok=True
)


print(
    f"✅ Output directory:\n"
    f"   {RECOVER_OUTPUT_PATH}"
)

print()


# ==============================================================================
# STEP 19 — SAVE FULL MERGED MODEL
# ==============================================================================

print("=" * 80)
print("📦 SAVING FULL MERGED MODEL")
print("=" * 80)
print()


try:

    merged_model.save_pretrained(
        RECOVER_OUTPUT_PATH,
        safe_serialization=True,
        max_shard_size="2GB"
    )

except Exception as e:

    raise RuntimeError(
        "❌ FAILED TO SAVE MERGED MODEL.\n\n"
        f"Output:\n{RECOVER_OUTPUT_PATH}\n\n"
        f"Error:\n{e}"
    )


print("✅ Model weights saved.")
print()


# ==============================================================================
# STEP 20 — SAVE TOKENIZER
# ==============================================================================

try:

    tokenizer.save_pretrained(
        RECOVER_OUTPUT_PATH
    )

    print(
        "✅ Tokenizer saved."
    )

except Exception as e:

    print(
        f"⚠️ Tokenizer save warning: {e}"
    )


# ==============================================================================
# STEP 21 — FINAL FILE INVENTORY
# ==============================================================================

print()
print("=" * 80)
print("📋 FINAL OUTPUT FILES")
print("=" * 80)
print()


saved_files = []


for root, dirs, files in os.walk(
    RECOVER_OUTPUT_PATH
):

    for filename in files:

        full_path = os.path.join(
            root,
            filename
        )

        relative_path = os.path.relpath(
            full_path,
            RECOVER_OUTPUT_PATH
        )

        saved_files.append(
            relative_path
        )


saved_files.sort()


for filename in saved_files:

    print(
        f"  ✅ {filename}"
    )


# ==============================================================================
# STEP 22 — FINAL SUCCESS MESSAGE
# ==============================================================================

print()
print("=" * 80)
print("🏆 PARTIALLY MERGED MODEL CREATED SUCCESSFULLY")
print("=" * 80)
print()

print(
    f"🤖 Base Model       : {RECOVER_MODEL_NAME}"
)

print(
    f"📊 Dataset          : {RECOVER_DATASET_NAME}"
)

print(
    f"📈 Checkpoint Step  : {TRAINED_GLOBAL_STEP}"
)

if estimated_samples is not None:

    print(
        f"📦 Approx. Samples  : {estimated_samples}"
    )

print(
    f"💾 Merge Source     : {SOURCE_PATH}"
)

print(
    f"📂 Final Model      : {RECOVER_OUTPUT_PATH}"
)

print()

print("✅ LoRA merged into the base model.")
print("✅ Full model files saved.")
print("✅ Tokenizer saved.")
print("✅ Output is LOCAL Colab storage.")
print("✅ Google Drive training checkpoints were untouched.")
print("❌ No GGUF conversion performed.")

print()
print("=" * 80)
print("🎉 RECOVERY COMPLETE")
print("=" * 80)

> **⚠️ Double Check File Path**

In [ ]:
# @title 🛠️ All-in-One GGUF Converter & Quantizer Form
# @markdown Fill in your parameters below and click the **Run (Play)** button! 🎯

import os

# @markdown **1. Select Format Type:**
format_type = "auto"  # @param ["auto", "f16", "f32", "Q4_K_M", "Q5_K_M", "Q8_0", "Q4_K_S", "Q5_K_S", "Q6_K", "Q3_K_M", "Q3_K_S", "Q2_K"]

# @markdown **2. Enter AI Model / Folder Name Path:**
input_folder_path = "/content/merged_16bit_model"  # @param {type:"string"}

# @markdown **3. Output GGUF File Path:**
output_file_path = "/content/merged_16bit_model.gguf"  # @param {type:"string"}


# ========================================================
# STEP 1: AUTOMATIC DEPENDENCY SETUP
# ========================================================
print("📦 Step 1: Checking & setting up llama.cpp dependencies...")

if not os.path.exists("llama.cpp"):
  !git clone https://github.com/ggerganov/llama.cpp.git
  !pip install -r llama.cpp/requirements.txt -q
  !make -C llama.cpp llama-quantize -j
else:
  # Ensure python requirements and quantize binary exist
  !pip install -r llama.cpp/requirements.txt -q
  if not os.path.exists("./llama.cpp/llama-quantize") and not os.path.exists(
      "./llama.cpp/build/bin/llama-quantize"
  ):
    !make -C llama.cpp llama-quantize -j

print("✅ Setup complete!\n")


# ========================================================
# STEP 2: CONVERSION & QUANTIZATION LOGIC
# ========================================================
if not os.path.exists(input_folder_path):
  print(
      f"❌ Error: The directory '{input_folder_path}' does not exist! Please"
      " check your path."
  )
else:
  # Detect conversion script name
  convert_script = "llama.cpp/convert_hf_to_gguf.py"
  if not os.path.exists(convert_script):
    convert_script = "llama.cpp/convert-hf-to-gguf.py"

  # Detect binary location
  quant_bin = "./llama.cpp/llama-quantize"
  if not os.path.exists(quant_bin):
    quant_bin = "./llama.cpp/build/bin/llama-quantize"

  # CASE A: Direct conversion (auto, f16, f32)
  if format_type in ["auto", "f16", "f32"]:
    print(f"🚀 Converting directly to GGUF ({format_type})...")
    !python {convert_script} "{input_folder_path}" --outfile "{output_file_path}" --outtype {format_type}
    print(f"\n✨ Done! Your model is saved at: {output_file_path}")

  # CASE B: Quantized formats (Q4_K_M, Q5_K_M, Q8_0, etc.)
  else:
    temp_gguf = "/content/temp_f16_conversion.gguf"

    print("⚙️ Step 2A: Converting model to temporary FP16 GGUF...")
    !python {convert_script} "{input_folder_path}" --outfile "{temp_gguf}" --outtype f16

    print(
        f"\n⚡ Step 2B: Quantizing model to {format_type} using"
        " llama-quantize..."
    )
    !{quant_bin} "{temp_gguf}" "{output_file_path}" {format_type}

    print("\n🧹 Step 2C: Cleaning up temporary files...")
    if os.path.exists(temp_gguf):
      os.remove(temp_gguf)

    print(
        f"\n🎉 Woohoo! Your {format_type} GGUF model is ready for LM Studio /"
        f" Ollama at:\n👉 {output_file_path}"
    )

**⚡ Optional: Upload Merged Model to Hugging Face Hub**

> **⚠️ Double Check File Path**

In [ ]:
# @title ⬇📤 2. Upload to Hugging Face Hub
# ==============================================================================
# LIGHTWEIGHT HF UPLOADER (NO UNSLOTH NEEDED! 🚀)
# ==============================================================================
!pip install --quiet huggingface_hub

from huggingface_hub import HfApi
import os

# 1. CONFIGURATION PANEL 🎛️
HF_WRITE_TOKEN = "hf_xxxxxxxxxxxxxxxxxxxxxxxxxx"  #@param {type:"string"}
HF_REPO_NAME = "your-name/my-finetuned-model"        #@param {type:"string"}
FOLDER_PATH = "/content/merged_16bit_model"             #@param {type:"string"}
REPO_VISIBILITY = "public"                             #@param ["private", "public"]

# 2. VERIFY FOLDER EXISTS
if not os.path.exists(FOLDER_PATH):
    raise FileNotFoundError(f"❌ Cannot find folder at: {FOLDER_PATH}")

# 3. INITIALIZE HF API & CREATE REPO
api = HfApi(token=HF_WRITE_TOKEN)

print(f"📁 Preparing Hugging Face repository '{HF_REPO_NAME}' ({REPO_VISIBILITY.upper()})...")
api.create_repo(
    repo_id=HF_REPO_NAME,
    private=(REPO_VISIBILITY == "private"),
    exist_ok=True,
    repo_type="model"
)

# 4. PUSH FOLDER TO HUGGING FACE
print(f"🚀 Uploading all files from '{FOLDER_PATH}'...")
api.upload_folder(
    folder_path=FOLDER_PATH,
    repo_id=HF_REPO_NAME,
    repo_type="model",
)

print(f"\n🎉 BOOM! Your model is live at: https://huggingface.co/{HF_REPO_NAME}")

#🛠️ Install hugging face tool

*   **pip install huggingface_hub**

or,
*   **pip install huggingface_hub --break-system-packages**


#📝 Command prompt

**</>**  **hf download your-username/my-finetuned-model  --local-dir /path/to/dir/[Folder where you will save files]/**

> **⚠️ Double Check File Path**



In [ ]:
# @title ⬇️ 3. Auto-Download Model (Browser Zip Download)
# ==============================================================================
import os
from google.colab import files

model_folder = "/content/merged_16bit_model" #@param {type:"string"}

if os.path.exists(model_folder):
    print("📦 Compressing model folder...")
    !zip -r /content/model.zip {model_folder}
    print("⬇️ Triggering browser download now...")
    files.download("/content/model.zip")
else:
    print(f"❌ Folder missing at {model_folder}! Please verify training export.")

---
**🪄 Single Line Command Prompt to make it ready to chat in Ollama or LM Studio**
---

🎯 **python convert_hf_to_gguf.py "/path/to/dir/[Folder Name Where All Files Saved]" --outfile "/path/to/dir/[AI Model Name].gguf" --outtype auto**

or,

🎯 **./venv/bin/python convert_hf_to_gguf.py "/path/to/dir/[Folder Name Where All Files Saved]" --outfile "/path/to/dir/[AI Model Name].gguf" --outtype auto**

---

**👇 If you want Q4_K_M or anyother format**
---

🎯 **python convert_hf_to_gguf.py "/path/to/dir/[Folder Name Where All Files Saved]" --outfile "/path/to/dir/[Temp Name].gguf" --outtype f16 && ./llama-quantize "/path/to/dir/[Temp Name].gguf" "/path/to/dir/[Final Name].gguf" Q4_K_M && rm "/path/to/dir/[Temp Name].gguf"**

or,

🎯 **./venv/bin/python convert_hf_to_gguf.py "/path/to/dir/[Folder Name Where All Files Saved]" --outfile "/path/to/dir/[Temp Name].gguf" --outtype f16 && ./build/bin/llama-quantize "/path/to/dir/[Temp Name].gguf" "/path/to/dir/[Final Name].gguf" Q4_K_M && rm "/path/to/dir/[Temp Name].gguf"**

---

In [ ]:
# @title 🔗 If you just want to connect to Google Drive.
print("📂 [1/10] Connecting Google Drive for Checkpoints & Export...")
DRIVE_BASE_DIR = "/content/drive/MyDrive/"
just_run_dont_worry_about_box = ""     #@param {type:"boolean"}
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print("✅ Google Drive connected successfully!")
except Exception as e:
    print(f"⚠️ Google Drive not mounted ({e}).")
